# Simple: Identity Field Analysis with PAMOLA.CORE

**Goal**: Learn identity field profiling basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Analyze identifier consistency and distribution using execute()
- Understand cross-matching and privacy implications
- Identify inconsistencies in identifier assignments

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path
- Verifies all imports work correctly

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── profiling/analyzers/
        └── 01_identity_analyzer_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.profiling.analyzers.identity import IdentityAnalysisOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Load data from `examples/data_examples/sample.csv`
- Auto-create sample data if file doesn't exist
- Inspect the dataset structure

**Expected output:** DataFrame with UIDs, entity IDs, and reference fields (name, DOB)

In [ ]:
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'

if not data_path.exists():
    print("⚠️  File not found, creating sample data...")
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample data with identifiers and reference fields
    sample_data = pd.DataFrame({
        'resume_id': range(1, 26),
        'UID': ['UID001', 'UID001', 'UID001', 'UID002', 'UID002',
               'UID003', 'UID003', 'UID003', 'UID003', 'UID004',
               'UID004', 'UID005', 'UID006', 'UID006', 'UID007',
               'UID008', 'UID008', 'UID009', 'UID009', 'UID009',
               'UID010', 'UID011', 'UID012', 'UID012', 'UID013'],
        'first_name': ['Alice', 'Alice', 'Alice', 'Bob', 'Bob',
                      'Charlie', 'Charlie', 'Charlie', 'Charlie', 'Diana',
                      'Diana', 'Eve', 'Frank', 'Frank', 'Grace',
                      'Henry', 'Henry', 'Iris', 'Iris', 'Iris',
                      'Jack', 'Karen', 'Leo', 'Leo', 'Mia'],
        'last_name': ['Smith', 'Smith', 'Smith', 'Jones', 'Jones',
                     'Brown', 'Brown', 'Brown', 'Brown', 'Prince',
                     'Prince', 'Adams', 'Miller', 'Miller', 'Lee',
                     'Wilson', 'Wilson', 'Chen', 'Chen', 'Chen',
                     'Taylor', 'White', 'Martin', 'Martin', 'Davis'],
        'birth_date': ['1985-03-15', '1985-03-15', '1985-03-15', '1990-07-22', '1990-07-22',
                      '1978-11-30', '1978-11-30', '1978-11-30', '1978-11-30', '1995-01-10',
                      '1995-01-10', '1982-05-18', '1988-09-25', '1988-09-25', '1992-12-05',
                      '1975-04-14', '1975-04-14', '1998-08-20', '1998-08-20', '1998-08-20',
                      '1980-02-28', '1987-06-17', '1993-10-08', '1993-10-08', '1976-03-22'],
        'gender': ['F', 'F', 'F', 'M', 'M',
                  'M', 'M', 'M', 'M', 'F',
                  'F', 'F', 'M', 'M', 'F',
                  'M', 'M', 'F', 'F', 'F',
                  'M', 'F', 'M', 'M', 'F']
    })
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created")

df = read_csv(data_path)
print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 10 records:")
print(df.head(10))

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:20s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

## Step 3: Setup Task Environment

**How to use:**
- Create task directory for outputs
- Initialize TaskReporter for tracking
- Setup DataSource with our DataFrame
- Configure progress tracker
- **Configure UID field and reference fields**

**Configuration pattern (production-style):**
```python
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "uid_field": "location_province",            # Identifier to analyze
        "reference_fields": [          # Fields defining identity
            "first_name",
            "last_name",
        ],
        "id_field": "resume_id"        # Optional entity ID
    }
```

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "uid_field": "location_province",              # Identifier to analyze - CUSTOMIZE THIS!
        "reference_fields": [            # Fields defining identity - CUSTOMIZE THIS!
            "first_name",
            "last_name",
        ],
        "id_field": "resume_id"          # Optional entity ID - CUSTOMIZE THIS!
    }

kwargs = create_config_kwargs()
uid_field = kwargs["uid_field"]
reference_fields = kwargs["reference_fields"]
id_field = kwargs["id_field"]

# Validate fields exist
print(f"\n🔍 Validating field configuration...\n")

# Validate UID field
if uid_field not in df.columns:
    raise ValueError(
        f"❌ UID field '{uid_field}' not found in dataset!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'uid_field' in create_config_kwargs()"
    )

# Validate reference fields
missing_refs = [f for f in reference_fields if f not in df.columns]
valid_refs = [f for f in reference_fields if f in df.columns]
if not valid_refs:
    raise ValueError(
        f"❌ No reference fields found in dataset!\n"
        f"Looking for: {', '.join(reference_fields)}\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'reference_fields' in create_config_kwargs()"
    )

if missing_refs:
    print(f"⚠️  Warning: Some reference fields not found: {', '.join(missing_refs)}")

# Validate ID field (optional)
if id_field and id_field not in df.columns:
    print(f"⚠️  Warning: ID field '{id_field}' not found. Distribution analysis will be skipped.")
    id_field = None

print(f"✓ UID field: '{uid_field}'")
print(f"  Unique values: {df[uid_field].nunique()}")
print(f"  Sample values: {list(df[uid_field].unique()[:5])}")

print(f"\n✓ Reference fields ({len(valid_refs)}):")
for field in valid_refs:
    print(f"  - {field:20s}: {df[field].nunique()} unique values")

if id_field:
    print(f"\n✓ Entity ID field: '{id_field}'")
    print(f"  Unique values: {df[id_field].nunique()}")

# Create task directory
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_001",
    task_type="identity_analysis",
    description=f"Identity consistency analysis for '{uid_field}'",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(dataframes={"main_dataset": df})
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Analyzing {uid_field} identity",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Create IdentityAnalysisOperation with full config
- Use `operation.execute()` with explicit execution configs
- Monitor progress through tracker

**Key parameters:**
- `uid_field=uid_field`: Primary identifier field to analyze
- `reference_fields=reference_fields`: Identity-defining fields
- `id_field='resume_id'`: Entity-level identifier (optional)
- `top_n=15`: Number of top examples to include in results
- `check_cross_matches=True`: Analyze cross-matching issues
- `min_similarity=0.8`: Threshold for fuzzy matching (0-1)
- `fuzzy_matching=False`: Use exact or fuzzy matching
- `generate_visualization=True`: Create charts
- `save_output=True`: Save to files
- `force_recalculation=False`: Use cache if available

In [ ]:
# Create operation with production-style configuration
operation = IdentityAnalysisOperation(
    # Core parameters
    uid_field=uid_field,                # Primary identifier
    reference_fields=reference_fields,  # Identity-defining fields
    id_field=id_field,                  # Optional entity ID
    
    # Analysis parameters
    top_n=15,                           # Top examples to include
    check_cross_matches=True,           # Analyze cross-matching
    min_similarity=0.8,                 # Fuzzy match threshold
    fuzzy_matching=False,               # Use exact matching
    
    # Processing settings
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,        # Create visualization artifacts
    save_output=True,                   # Save results to files
    force_recalculation=False           # Use cache when available
)

print("✓ Operation configured")
print(f"  UID field:            {operation.uid_field}")
print(f"  Reference fields:     {len(operation.reference_fields)}")
print(f"  Entity ID field:      {operation.id_field}")
print(f"  Top N:                {operation.top_n}")
print(f"  Check cross-matches:  {operation.check_cross_matches}")
print(f"  Fuzzy matching:       {operation.fuzzy_matching}")
print(f"  Visualizations:       {operation.generate_visualization}")
print(f"  Save output:          {operation.save_output}")
print(f"  Force recalc:         {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing identity analysis...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Load the analysis metrics from task_dir
- Review consistency statistics
- Examine cross-matching issues

**Generated artifacts:**
- Multiple metrics JSONs in metrics/
- Visualizations in visualizations/

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load metrics files
metrics_dir = task_dir / 'metrics'
if metrics_dir.exists():
    print("\n" + "=" * 80)
    print("📊 IDENTITY ANALYSIS RESULTS")
    print("=" * 80)
    
    # Load identifier statistics
    identifier_files = list(metrics_dir.glob('*identifier_metrics*.json'))
    if identifier_files:
        with open(identifier_files[0], 'r') as f:
            identifier_stats = json.load(f)
        
        print("\n📈 Identifier Statistics:")
        print(f"  Total records:          {identifier_stats.get('total_records', 0):,}")
        print(f"  Unique identifiers:     {identifier_stats.get('unique_identifiers', 0):,}")
        print(f"  Null identifiers:       {identifier_stats.get('null_identifiers', 0):,}")
        print(f"  Coverage percentage:    {identifier_stats.get('coverage_percentage', 0):.2f}%")
        print(f"  Uniqueness ratio:       {identifier_stats.get('uniqueness_ratio', 0):.4f}")
        
        # Relationship statistics if available
        if 'one_to_one_count' in identifier_stats:
            print(f"\n  Relationship Metrics:")
            print(f"    1-to-1 relationships:   {identifier_stats.get('one_to_one_count', 0):,}")
            print(f"    1-to-N relationships:   {identifier_stats.get('one_to_many_count', 0):,}")
            print(f"    Avg entities per ID:    {identifier_stats.get('avg_entities_per_id', 0):.2f}")
            print(f"    Max entities per ID:    {identifier_stats.get('max_entities_per_id', 0)}")
    
    # Load consistency analysis
    consistency_files = list(metrics_dir.glob('*consistency_metrics*.json'))
    if consistency_files:
        with open(consistency_files[0], 'r') as f:
            consistency_stats = json.load(f)
        
        print("\n🔍 Consistency Analysis:")
        print(f"  Total combinations:     {consistency_stats.get('total_combinations', 0):,}")
        print(f"  Consistent:             {consistency_stats.get('consistent_combinations', 0):,}")
        print(f"  Inconsistent:           {consistency_stats.get('inconsistent_combinations', 0):,}")
        print(f"  Match percentage:       {consistency_stats.get('match_percentage', 0):.2f}%")
        print(f"  Reference fields used:  {', '.join(consistency_stats.get('reference_fields_used', []))}")
        
        # Show mismatch examples
        if consistency_stats.get('mismatch_examples'):
            print(f"\n  Top Mismatches:")
            for i, example in enumerate(consistency_stats['mismatch_examples'][:3], 1):
                print(f"\n    {i}. Reference values:")
                for k, v in example['reference_values'].items():
                    print(f"       {k}: {v}")
                print(f"       Multiple UIDs: {len(example['id_values'])} ({', '.join(example['id_values'][:3])})")
                print(f"       Record count: {example['count']}")
    
    # Load distribution analysis
    distribution_files = list(metrics_dir.glob('*distribution_metrics*.json'))
    if distribution_files:
        with open(distribution_files[0], 'r') as f:
            distribution_stats = json.load(f)
        
        print("\n📊 Distribution Analysis:")
        print(f"  Total identifiers:      {distribution_stats.get('total_identifiers', 0):,}")
        print(f"  Average count:          {distribution_stats.get('avg_count', 0):.2f}")
        print(f"  Median count:           {distribution_stats.get('median_count', 0):.2f}")
        print(f"  Max count:              {distribution_stats.get('max_count', 0)}")
    
    # Load cross-match analysis
    cross_match_files = list(metrics_dir.glob('*cross_match_metrics*.json'))
    if cross_match_files:
        with open(cross_match_files[0], 'r') as f:
            cross_match_stats = json.load(f)
        
        print("\n⚠️  Cross-Match Analysis:")
        print(f"  Total cross-matches:    {cross_match_stats.get('total_cross_matches', 0):,}")
        print(f"  Reference fields used:  {', '.join(cross_match_stats.get('reference_fields_used', []))}")
        
        if cross_match_stats.get('cross_match_examples'):
            print(f"\n  Top Cross-Match Examples:")
            for i, example in enumerate(cross_match_stats['cross_match_examples'][:3], 1):
                print(f"\n    {i}. Same person, different UIDs:")
                for k, v in example['reference_values'].items():
                    print(f"       {k}: {v}")
                print(f"       UIDs assigned: {', '.join(example['id_values'])}")
                print(f"       Record count: {example['count']}")
    
    # Display result metrics summary
    print("\n" + "=" * 80)
    print("✨ SUMMARY")
    print("=" * 80)
    if result.metrics:
        for key, value in list(result.metrics.items())[:10]:
            if isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:30s}: {value:.4f}")
                else:
                    print(f"  {key:30s}: {value}")
    
    print(f"\n💡 Identity field '{uid_field}' successfully analyzed!")
else:
    print("⚠️  No metrics directory found")

## Step 6: Review Artifacts Location

**How to use:**
- Check all generated files
- Navigate to directories for manual inspection

**Output structure:**
```
examples/data_examples/simple_output/
├── metrics/          # Multiple analysis metrics JSONs
├── visualizations/   # PNG charts
└── config/           # Operation config
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

for subdir in ['metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")

print("\n✅ All artifacts saved successfully!")

## Step 7: Detailed Artifact Review

**How to use:**
- Review all generated artifacts in detail
- Automatically loads the NEWEST files from each category
- Displays visualizations inline for easy review

**What you'll see:**
1. **Identifier Metrics**: Basic identifier statistics
2. **Consistency Metrics**: Identifier-reference field consistency
3. **Distribution Metrics**: Records per identifier distribution
4. **Cross-Match Metrics**: Cross-matching issues
5. **Visualizations**: Charts for all analyses

**Note:** Only the most recent files are shown to avoid confusion from multiple runs

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

metrics_dir = task_dir / 'metrics'
if metrics_dir.exists():
    # 1️⃣ IDENTIFIER METRICS (NEWEST)
    print("\n1️⃣ IDENTIFIER METRICS:")
    print("=" * 80)

    identifier_files = sorted(
        metrics_dir.glob("*identifier_metrics*.json"),
        key=lambda x: x.stat().st_mtime,
        reverse=True,
    )

    if not identifier_files:
        print("⚠️  No identifier metrics found")
    else:
        latest_file = identifier_files[0]
        print(f"📄 Loading: {latest_file.name}")

        with open(latest_file, "r", encoding="utf-8") as f:
            raw = json.load(f)

        metadata = raw.get("metadata", {})
        metrics = raw.get("metrics", {})

        # METADATA
        print("\n🧾 METADATA")
        print("-" * 80)
        print(f"Timestamp  : {metadata.get('timestamp', 'N/A')}")
        print(f"Operation  : {metadata.get('name', 'N/A')}")
        op = metadata.get("operation", {})
        print(f"Module     : {op.get('module', 'N/A')}")
        print(f"Class      : {op.get('class', 'N/A')}")

        # CORE IDENTIFIER METRICS
        print("\n📊 CORE IDENTIFIER METRICS")
        print("-" * 80)

        print(f"Total records          : {metrics.get('total_records', 0):,}")
        print(f"Unique identifiers     : {metrics.get('unique_identifiers', 0):,}")
        print(f"Null identifiers       : {metrics.get('null_identifiers', 0):,}")
        print(f"Coverage percentage    : {metrics.get('coverage_percentage', 0):.2f}%")
        print(f"Uniqueness ratio       : {metrics.get('uniqueness_ratio', 0):.6f}")

        # CARDINALITY ANALYSIS
        print("\n🔗 CARDINALITY ANALYSIS")
        print("-" * 80)

        print(f"One-to-one count       : {metrics.get('one_to_one_count', 0):,}")
        print(f"One-to-many count      : {metrics.get('one_to_many_count', 0):,}")
        print(f"Avg entities per ID    : {metrics.get('avg_entities_per_id', 0):.2f}")
        print(f"Max entities per ID    : {metrics.get('max_entities_per_id', 0):,}")
        print(f"Avg IDs per entity     : {metrics.get('avg_ids_per_entity', 0):.2f}")
        print(f"Max IDs per entity     : {metrics.get('max_ids_per_entity', 0):,}")

        # INTERPRETATION
        print("\n🧠 INTERPRETATION")
        print("-" * 80)

        uniq_ratio = metrics.get("uniqueness_ratio", 0)
        one_to_many = metrics.get("one_to_many_count", 0)

        if uniq_ratio < 0.01 and one_to_many > 0:
            print("⚠️  Identifier has VERY LOW uniqueness")
            print("   → Acts as a QUASI / GROUPING identifier, not a direct identifier")
            print("   → High re-identification risk only when combined with other fields")
        elif uniq_ratio > 0.5:
            print("🚨 High uniqueness identifier")
            print("   → Direct identifier risk")
        else:
            print("ℹ️  Moderate identifier strength")

        print("\n   Recommended usage:")
        print("   • Safe for aggregation / grouping")
        print("   • NOT suitable as anonymized entity key")
    
    # 2. CONSISTENCY METRICS
    print("\n\n2️⃣ CONSISTENCY METRICS:")
    print("-" * 80)
    
    consistency_files = sorted(
        metrics_dir.glob("*consistency_metrics*.json"),
        key=lambda x: x.stat().st_mtime,
        reverse=True,
    )

    if not consistency_files:
        print("⚠️  No consistency metrics found")
    else:
        latest_file = consistency_files[0]
        print(f"📄 Loading: {latest_file.name}")

        with open(latest_file, "r", encoding="utf-8") as f:
            raw = json.load(f)

        metadata = raw.get("metadata", {})
        metrics = raw.get("metrics", {})

        # METADATA
        print("\n🧾 METADATA")
        print("-" * 80)
        print(f"Timestamp  : {metadata.get('timestamp', 'N/A')}")
        print(f"Operation  : {metadata.get('name', 'N/A')}")
        op = metadata.get("operation", {})
        print(f"Module     : {op.get('module', 'N/A')}")
        print(f"Class      : {op.get('class', 'N/A')}")
        print(f"References : {', '.join(metrics.get('reference_fields_used', []))}")

        # CORE CONSISTENCY METRICS
        print("\n📊 CONSISTENCY RESULTS")
        print("-" * 80)

        total_records = metrics.get("total_records", 0)
        total_combos = metrics.get("total_combinations", 0)
        consistent = metrics.get("consistent_combinations", 0)
        inconsistent = metrics.get("inconsistent_combinations", 0)
        match_pct = metrics.get("match_percentage", 0.0)

        print(f"Total records            : {total_records:,}")
        print(f"Total combinations       : {total_combos:,}")
        print(f"Consistent combinations  : {consistent:,}")
        print(f"Inconsistent combinations: {inconsistent:,}")
        print(f"Match percentage         : {match_pct:.2f}%")

        # MISMATCH ANALYSIS
        mismatch_examples = metrics.get("mismatch_examples", [])

        if mismatch_examples:
            print("\n❌ SAMPLE MISMATCHES")
            print("-" * 80)

            for i, ex in enumerate(mismatch_examples[:5], 1):
                refs = ex.get("reference_values", {})
                ids = ex.get("id_values", [])
                count = ex.get("count", 0)

                ref_str = ", ".join(f"{k}={v}" for k, v in refs.items())
                print(f"{i}. {ref_str}")
                print(f"   Distinct ID values: {len(ids)}")
                print(f"   ID examples       : {ids[:6]}")
                print(f"   Record count      : {count}")

        # INTERPRETATION
        print("\n🧠 INTERPRETATION")
        print("-" * 80)

        if match_pct < 20:
            print("⚠️  VERY LOW consistency detected")
            print("   → Identifier is NOT functionally dependent on reference fields")
            print("   → Acts as a MANY-TO-MANY attribute")
            print("   → Strong signal this is a QUASI / CONTEXTUAL attribute")
        elif match_pct < 60:
            print("ℹ️  Partial consistency detected")
            print("   → Identifier may be context-dependent")
        else:
            print("✅ High consistency")
            print("   → Identifier is stable with respect to reference fields")

        print("\n📌 PRIVACY IMPLICATIONS")
        print("   • Unsafe as a person identifier")
        print("   • Safe for aggregation & regional analysis")
        print("   • Should be generalized or bucketed if combined with PII")
    
    # 3. DISTRIBUTION METRICS
    print("\n\n3️⃣ DISTRIBUTION METRICS:")
    print("-" * 80)
    
    distribution_files = sorted(
        metrics_dir.glob("*distribution_metrics*.json"),
        key=lambda x: x.stat().st_mtime,
        reverse=True,
    )

    if not distribution_files:
        print("⚠️  No distribution metrics found")
    else:
        latest_file = distribution_files[0]
        print(f"📄 Loading: {latest_file.name}")

        with open(latest_file, "r", encoding="utf-8") as f:
            raw = json.load(f)

        metadata = raw.get("metadata", {})
        metrics = raw.get("metrics", {})

        # METADATA
        print("\n🧾 METADATA")
        print("-" * 80)
        print(f"Timestamp : {metadata.get('timestamp', 'N/A')}")
        print(f"Operation : {metadata.get('name', 'N/A')}")
        op = metadata.get("operation", {})
        print(f"Module    : {op.get('module', 'N/A')}")
        print(f"Class     : {op.get('class', 'N/A')}")

        # CORE DISTRIBUTION STATS
        print("\n📊 DISTRIBUTION STATS")
        print("-" * 80)

        total_ids = metrics.get("total_identifiers", 0)
        total_records = metrics.get("total_records", 0)
        avg_count = metrics.get("avg_count", 0.0)
        median_count = metrics.get("median_count", 0.0)
        max_count = metrics.get("max_count", 0)
        min_count = metrics.get("min_count", 0)

        print(f"Total identifiers : {total_ids:,}")
        print(f"Total records     : {total_records:,}")
        print(f"Average count     : {avg_count:.2f}")
        print(f"Median count      : {median_count:.2f}")
        print(f"Max count         : {max_count:,}")
        print(f"Min count         : {min_count:,}")

        # DISTRIBUTION SHAPE
        dist = metrics.get("distribution", {})
        if dist:
            print("\n📈 DISTRIBUTION SHAPE (count → #identifiers)")
            print("-" * 80)

            for count, freq in sorted(dist.items(), key=lambda x: int(x[0])):
                print(f"{int(count):>5} → {freq}")

        # TOP IDENTIFIER EXAMPLES
        top_examples = metrics.get("top_examples", [])
        if top_examples:
            print("\n🔝 TOP IDENTIFIERS")
            print("-" * 80)

            for i, ex in enumerate(top_examples[:5], 1):
                identifier = ex.get("identifier")
                count = ex.get("count", 0)
                entities = ex.get("entities", [])

                print(f"{i}. Identifier : {identifier}")
                print(f"   Count      : {count}")
                print(f"   Entities   : {len(entities)}")
                if ex.get("sample"):
                    print(f"   Sample     : {ex['sample']}")

        # INTERPRETATION
        print("\n🧠 INTERPRETATION")
        print("-" * 80)

        if total_ids < 20 and avg_count > 50:
            print("⚠️  Very low cardinality & high reuse detected")
            print("   → Attribute is categorical, NOT identifying")
            print("   → Strong MANY-TO-ONE behavior")
        elif max_count / max(total_records, 1) > 0.3:
            print("⚠️  Heavy skew detected")
            print("   → One category dominates the population")
        else:
            print("ℹ️  Balanced categorical distribution")

        print("\n📌 PRIVACY IMPLICATIONS")
        print("   • Safe as standalone attribute")
        print("   • Low re-identification power")
        print("   • Risk only arises when combined with PII")
    
    # 4. CROSS-MATCH METRICS
    print("\n\n4️⃣ CROSS-MATCH METRICS:")
    print("-" * 80)
    
    cross_match_files = sorted(
        metrics_dir.glob("*cross_match_metrics*.json"),
        key=lambda x: x.stat().st_mtime,
        reverse=True,
    )

    if not cross_match_files:
        print("⚠️  No cross-match metrics found")
    else:
        latest_file = cross_match_files[0]
        print(f"📄 Loading: {latest_file.name}")

        with open(latest_file, "r", encoding="utf-8") as f:
            raw = json.load(f)

        metadata = raw.get("metadata", {})
        metrics = raw.get("metrics", {})

        # METADATA
        print("\n🧾 METADATA")
        print("-" * 80)
        print(f"Timestamp : {metadata.get('timestamp', 'N/A')}")
        print(f"Operation : {metadata.get('name', 'N/A')}")
        op = metadata.get("operation", {})
        print(f"Module    : {op.get('module', 'N/A')}")
        print(f"Class     : {op.get('class', 'N/A')}")

        # CORE CROSS-MATCH METRICS
        print("\n📊 CROSS-MATCH RESULTS")
        print("-" * 80)

        total_records = metrics.get("total_records", 0)
        total_cross_matches = metrics.get("total_cross_matches", 0)
        ref_fields = metrics.get("reference_fields_used", [])

        print(f"Total records        : {total_records:,}")
        print(f"Total cross-matches  : {total_cross_matches:,}")
        print(f"Reference fields     : {', '.join(ref_fields) if ref_fields else 'N/A'}")

        # CROSS-MATCH RATE
        cross_match_rate = (
            total_cross_matches / total_records * 100
            if total_records else 0
        )

        print(f"Cross-match rate     : {cross_match_rate:.2f}%")

        # SAMPLE CROSS-MATCH EXAMPLES
        examples = metrics.get("cross_match_examples", [])
        if examples:
            print("\n🔍 SAMPLE CROSS-MATCHES (Top 5)")
            print("-" * 80)

            for i, ex in enumerate(examples[:5], 1):
                refs = ex.get("reference_values", {})
                ids = ex.get("id_values", [])
                count = ex.get("count", 0)

                print(f"{i}. Reference values : {refs}")
                print(f"   Identifier values: {ids}")
                print(f"   Count             : {count}")

        # INTERPRETATION
        print("\n🧠 INTERPRETATION")
        print("-" * 80)

        if cross_match_rate > 20:
            print("⚠️  High cross-attribute inconsistency detected")
            print("   → Identifier strongly conflicts with reference fields")
            print("   → Indicates MANY-TO-MANY or data quality issues")
        elif cross_match_rate > 5:
            print("⚠️  Moderate cross-matching detected")
            print("   → Attribute weakly correlates with identity")
        else:
            print("ℹ️  Low cross-matching detected")
            print("   → Attribute largely independent")

        print("\n📌 PRIVACY IMPLICATIONS")
        print("   • Cross-matching does NOT imply identifiability")
        print("   • High cross-match often REDUCES identity confidence")
        print("   • Treat as contextual / quasi attribute")

# 5. VISUALIZATIONS
print("\n\n5️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        latest_time = viz_files[0].stat().st_mtime
        time_threshold = 10
        latest_viz_batch = [
            f for f in viz_files 
            if abs(f.stat().st_mtime - latest_time) < time_threshold
        ]
        
        latest_viz_batch.sort(key=lambda x: x.name)
        
        latest_mtime = datetime.fromtimestamp(latest_time)
        print(f"✓ Found {len(viz_files)} total visualization(s)")
        print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
        print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
        
        for i, viz_file in enumerate(latest_viz_batch, 1):
            # Extract readable name
            if 'identifier_stats' in viz_file.name:
                viz_name = "Identifier Statistics"
            elif 'consistency_stats' in viz_file.name:
                viz_name = "Consistency Analysis"
            elif 'distribution_analysis' in viz_file.name:
                viz_name = "Distribution Analysis"
            elif 'cross_match_bar' in viz_file.name:
                viz_name = "Cross-Match Analysis"
            else:
                viz_name = viz_file.stem.replace('_', ' ').title()
            
            print(f"\n{i}. {viz_name}")
            print(f"   File: {viz_file.name}")
            print("-" * 60)
            try:
                display(Image(filename=str(viz_file)))
            except Exception as e:
                print(f"   ⚠️  Could not display: {e}")
        
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**  
✅ Load data from examples/data_examples/  
✅ Setup environment with TaskReporter, DataSource, ProgressTracker  
✅ Configure identity analysis with UID and reference fields  
✅ Execute using operation.execute()  
✅ Analyze results and review artifacts  

**Key takeaways:**
- Identity analysis checks if UIDs consistently map to same people
- Reference fields (name, DOB, etc.) define a person's identity
- Consistency analysis finds UIDs with multiple identities
- Cross-matching finds same person with multiple UIDs
- Low consistency = identity confusion = privacy/quality issues
- Distribution analysis shows how records spread across UIDs
- High cross-matching = duplicate identities = data quality problem

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_identity_analyzer_advanced.ipynb`** includes:
- Fuzzy matching for similar but not identical records
- Custom similarity thresholds
- Multi-level identifier hierarchies
- Identifier resolution strategies
- Advanced privacy risk assessment

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)